# Field extraction: SkewerJAX against fake_spectra, one difference at a time

**Scope: the fields only.** Starting from the same TNG50-4 snapshot and the same sightlines, both codes extract what the optical depth needs from the cells: densities, temperatures, line widths, the ionisation state, and the geometry that puts the cells on the sightline. This notebook isolates every way fake_spectra's extraction differs from SkewerJAX's default, and measures each one. The integration of τ is the subject of `6_optical_depth_from_simulations_comparison_fake_spectra.ipynb`.

**The default pipeline** is explained in `3_fields_from_simulations.ipynb`.

**Method.**
1. fake_spectra's extraction is reproduced by `ss.FAKE_SPECTRA_EXTRACTION`, and its per-cell values are matched (§2).
2. Each difference is then one field of `ExtractionOptions`, switched alone (§3). For each: what fake_spectra does, the matched mode that recovers it, and its measured cost per cell and along the skewer (columns and temperatures per pixel).
3. §4 measures what each difference costs in the spectrum, with SkewerJAX's default integration held fixed. These are the numbers used for the final decomposition in the integration notebook.
4. §5 ranks them.

**fake_spectra.** With a build directory it runs live; otherwise every fake_spectra number is read from `FS_REFERENCE`.

**Runtime.** About 10 min from an empty cache; a few seconds cached.

## 0. Setup

In [ ]:
import os, sys, time, warnings
os.environ.setdefault("JAX_PLATFORMS", "cpu")             # float64 runs faster on this CPU than on a consumer GPU
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
from pathlib import Path
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib as mpl
import matplotlib.pyplot as plt
from skewerjax.config.setup_matplotlib import (matplotlib_tol_config, TOL_BRIGHT as TOL, ENTITY_COLORS as EC, MODE_STYLES as MS,
                                               tol_cmap, add_panel_letters, residual_band)
font, rcnew = matplotlib_tol_config()                         # the package style with Paul Tol's colour-blind-safe palette
plt.rc("font", **font); plt.rcParams.update(rcnew)

# ------------------------------------------------------------------ configuration (edit here)
REPO = Path("/home/dlopez/Documentos/0.profesional/Postdoc/USP/Projects/SkewerJAX")
DATA = REPO / "claude_agents/tmp_files/20261021_1335"        # caches and fake_spectra reference data (to be moved: see the notebook header)
SNAP_BASE, SNAPNUM = "/home/STORAGE/TNG50-4", 25          # read-only
CACHE = DATA / "cache" / "sjx_cache.h5"                    # SkewerJAX tau results (None: no cache)
DS_KMS = 0.25                                                # spacing of the sampling points, in Hubble velocity [km/s]
RES = 1.0                                                    # spectral pixel [km/s]
T_START = time.perf_counter()
from skewerjax.simulations import snapshot_readers as sr, gas_physics as gp, cell_selection as sel
from skewerjax.simulations import simulation_skewers as ss
from skewerjax.fields import deposition as dep
from skewerjax.optical_depth import absorbers as ab_
from skewerjax.optical_depth.optical_depth import OpticalDepthCalculator
from skewerjax.skewers.skewers import SkewerBundle, path_geometry, axis_aligned_skewer_points
from skewerjax.constants.lines import LineBundle
from skewerjax.constants.physical_constants import phys
print("jax", jax.__version__, jax.devices()[0], "| x64", jax.config.jax_enable_x64)

RESULTS = []
def check(name, value, tol, section, kind="pass", detail=""):
    """kind="pass": PASS if value <= tol; "explained": EXPLAINED if value <= tol (a difference accounted
    for by its parts); "expect": PASS if value > tol (the test must detect something)."""
    ok = bool(value > tol) if kind == "expect" else bool(value <= tol)
    tag = {"pass": ("PASS", "FAIL"), "expect": ("PASS", "FAIL"), "explained": ("EXPLAINED", "UNEXPLAINED")}[kind][0 if ok else 1]
    RESULTS.append(dict(section=section, name=name, value=float(value), tol=tol, kind=kind, tag=tag, detail=detail))
    print(f"[{tag}] {section:5s} {name}: {value:.3e} ({'>' if kind == 'expect' else '<='} {tol:g}) {detail}")
def rel(a, b, mask=None):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.ones(a.shape, bool) if mask is None else np.asarray(mask, bool)
    return np.abs(a[m] - b[m]) / np.maximum(np.abs(b[m]), 1e-300)

In [ ]:
# ------------------------------------------------------------------ fake_spectra: live (its build) or the reference file
from skewerjax.optical_depth.fake_spectra_reference import FakeSpectraSource, SKEWER_SETS
FAKE_SPECTRA_BUILD = REPO / "claude_agents/tmp_files/20261019_2155/fake_spectra_build"   # None: reference file only
FS_DRIVER = DATA / "run_fake_spectra_skewers.py"                                        # runs fake_spectra in its own process
FS_CACHE = DATA / "fs_cache"                                                             # its results (fs_*.h5, fs_products.h5)
FS_REFERENCE = DATA / "fake_spectra_reference_tng50-4_snap025.h5"                        # every fake_spectra number used here
EXPORT_REFERENCE = False                                                                  # True: add new fake_spectra results to FS_REFERENCE (live mode)
FS = FakeSpectraSource(FS_REFERENCE, build=FAKE_SPECTRA_BUILD, driver=FS_DRIVER, cache=FS_CACHE, snap_base=SNAP_BASE, snapnum=SNAPNUM, res=RES)
rd = sr.ArepoHDF5Reader(SNAP_BASE, SNAPNUM); H = rd.header; L = H.box_cMpc
FC = FS.constants()
LYA, LB_SI = LineBundle(["HI-LyA"], ref_line_id="HI-LyA"), LineBundle(["SiIII-1206"], ref_line_id="SiIII-1206")
SI = gp.Ion("Si", 3)
SK = {name: ss.SimulationSkewers.axis_aligned(rd, SKEWER_SETS[name]["cofm"], SKEWER_SETS[name]["axis"], DS_KMS, name=name, cache=CACHE)
      for name in ("single", "nset", "sk8")}
v, dv = SK["single"].spectral_grid(RES)                     # fake_spectra's periodic grid, shared by both codes
print("fake_spectra:", FS.mode, f"| spectral grid: {v.size} pixels of {dv:.6f} km/s")
for name, S in SK.items():
    print(f"{name:6s}: {S.n_skewers:2d} skewers x {S.points.shape[1]} points, {len(S.cells):,} cells selected in {S.selection_s:.1f} s")
E_FS, I_FS, E_DEF, I_DEF = ss.FAKE_SPECTRA_EXTRACTION, ss.FAKE_SPECTRA_INTEGRATION, ss.ExtractionOptions(), ss.IntegrationOptions()
EFFECT_STYLE = {                                             # one colour and line style per difference, in every figure
    "tautail": (TOL["grey"], "-"), "pixel average": (TOL["yellow"], ":"), "detuning": (TOL["cyan"], "-."), "7/8 normalisation": (TOL["green"], "-"),
    "chord quadrature": (TOL["yellow"], "--"), "frame": (TOL["purple"], "-"), "T constants": (TOL["cyan"], ":"), "b constants": (TOL["yellow"], "-."),
    "mass constant": (TOL["green"], "--"), "neutral fraction": (TOL["grey"], ":"), "CLOUDY density": (TOL["purple"], "--"),
    "CLOUDY interpolation": (TOL["red"], ":"), "deposition": (TOL["blue"], "-")}
def logbin(k, y, nb=40):
    """Mean of y in nb log-spaced k bins (figures only; every check uses all modes)."""
    e = np.geomspace(k[0], k[-1] * 1.0001, nb + 1); j = np.digitize(k, e) - 1; n = np.bincount(j, minlength=nb)[:nb]
    return np.sqrt(e[1:] * e[:-1])[n > 0], (np.bincount(j, weights=y, minlength=nb)[:nb] / np.maximum(n, 1))[n > 0]
FS_TAU = dict(single=FS.run("single_matched", "single", products="tau,colden,temp,velocity")["tau"], nset=FS.run("nset_default", "nset", sf_neutral=True)["tau"],
              sk8_SiIII=FS.run("nset_SiIII", "nset", species="Si:3:1206")["tau"][8:9])

## 1. The extraction differences

| field of `ExtractionOptions` | SkewerJAX default | fake_spectra (`FAKE_SPECTRA_EXTRACTION`) | what it changes |
|---|---|---|---|
| `mass_unit` | `"amu"`: n = ρX/(A m_u) | `"proton"`: n = ρX/(A m_p) | every density, by m_u/m_p |
| `T_constants` | `"codata"` m_p, k_B | `"fake_spectra"` m_p, k_B | T, by 8×10⁻⁶ |
| `b_constants` | b² = 2k_BT/(A m_u) | b² = 2kT/(A m_p) with the C engine's k | every line width |
| `x_HI_model` | `"stored_sf"`: stored x_HI, star-forming cells recomputed at n_H | `"fake_sf"`: stored, star-forming cells recomputed at ρ/m_p above fake_spectra's threshold | x_HI of star-forming cells |
| `cloudy_density` | `"hydrogen"`: hden = n_H | `"fake_spectra"`: hden = 0.774 ρ/m_p | metal ion fractions |
| `cloudy_interpolation` | `"linear"` in log f | `"cubic"` spline | metal ion fractions |
| `deposition` | `"voronoi"`: the cells of the mesh-generating points | `"tophat"`: a sphere of radius h = V^(1/3) per cell, density (3/4π) n | where the gas sits along the sightline |

Selection (which cells are considered) is not a difference: both codes take every cell that can reach the sightline (§3.5).

## 2. fake_spectra's per-cell values, reproduced

fake_spectra passes, for every cell, its position, velocity, ion density, temperature and size to its C engine (`Spectra._read_particle_data`). They are compared with `SimulationSkewers.properties(ion, FAKE_SPECTRA_EXTRACTION)` on the cells whose top-hat sphere touches the single skewer: the cells that fake_spectra itself uses.

In [ ]:
S = SK["single"]
TH1 = S.segments("tophat")
touch = np.unique(TH1.cell[TH1.cell >= 0])                                # cells whose sphere touches the skewer
fv = FS.cell_values(rd, S.cells, "single", "H", 1)
fvt = {k: (np.asarray(x)[touch] if np.ndim(x) else x) for k, x in fv.items()}
P_fs, P_def = S.properties(gp.HI, E_FS), S.properties(gp.HI, E_DEF)
sub = lambda P, k: np.asarray(P[k])[touch]
ax = int(SKEWER_SETS["single"]["axis"][0]) - 1
nH_f = fvt["rhoH"] * fvt["mass_frac"] / fvt["amumass"]
b_f = np.sqrt(2 * 1.3806504e-16 / (1.00794 * 1.67262178e-24) * fvt["temp"]) / 1e5         # absorption.cpp: its k, m_p
vel_t = np.asarray(S.cells.fields["velocity"])[touch, ax]
rows = [("n_H", sub(P_fs, "n_H"), sub(P_def, "n_H"), nH_f), ("T", sub(P_fs, "T"), sub(P_def, "T"), fvt["temp"]),
        ("x_HI", sub(P_fs, "ion_fraction"), sub(P_def, "ion_fraction"), fvt["ion_frac"]),
        ("n_HI", sub(P_fs, "n_ion"), sub(P_def, "n_ion"), fvt["elem_den"] / FC["rscale"]),
        ("v_pec along the skewer", vel_t, vel_t, fvt["vel"][:, ax]),
        ("b", np.sqrt(gp.FAKE_SPECTRA_B2_RATIO) * gp.doppler_b_kms(sub(P_fs, "T"), 1.00794), gp.doppler_b_kms(sub(P_def, "T"), 1.00794), b_f),
        ("size h = V^(1/3)", np.asarray(S.cells.size_cMpc)[touch], np.asarray(S.cells.size_cMpc)[touch], fvt["hh"] * H.code_length_cMpc)]
print(f"{touch.size} cells touch the single skewer | relative difference to fake_spectra's per-cell values")
print(f"{'quantity':24s} {'matched: max':>13s} {'default: median':>16s} {'default: max':>13s}")
for name, m, d, f in rows:
    print(f"{name:24s} {rel(m, f).max():13.2e} {np.median(rel(d, f)):16.4e} {rel(d, f).max():13.4e}")
    check(f"matched per-cell {name} == fake_spectra", rel(m, f).max(), 2e-6, "2")

With `FAKE_SPECTRA_EXTRACTION`, every per-cell value agrees with fake_spectra's to ≤ 2×10⁻⁶: fake_spectra keeps ρ and T in float32, which sets that floor. The default differs by the constants (n_H, b) and nothing else on these cells. The sections below take each difference in turn.

## 3. Each difference, one at a time

### 3.1 The mass constant (`mass_unit`)

**What fake_spectra does.** The number density of an element is ρX/(A m), with A its standard atomic weight. A is defined in atomic mass units, so m = A m_u. fake_spectra multiplies A by the **proton** mass (its constant labelled "1 a.m.u." is m_p), which is heavier by m_p/m_u = 1.007276.

**Its cost.**
- **Per cell:** every density of every element is 0.728% lower. Through the line width b ∝ m^(−1/2) (the `b_constants` item, §3.3), every line is also 0.36% narrower.
- **Along the skewer:** the same factor in every column.
- **Matched mode:** `mass_unit="proton"`.

In [ ]:
r_n = np.asarray(P_def["n_H"]) / np.asarray(S.properties(gp.HI, E_DEF.replace(mass_unit="proton"))["n_H"])
print(f"n_H(m_u) / n_H(m_p) = {np.median(r_n):.7f} (m_p/m_u = {phys.mp / gp.M_U_G:.7f})")
check("mass constant: per-cell density ratio == m_p/m_u", np.abs(r_n - phys.mp / gp.M_U_G).max(), 1e-12, "3.1")
N_def = S.column_per_pixel(gp.HI, E_DEF); N_mp = S.column_per_pixel(gp.HI, E_DEF.replace(mass_unit="proton"))
check("mass constant: column per pixel ratio == m_p/m_u", np.abs(N_def[N_mp > 0] / N_mp[N_mp > 0] - phys.mp / gp.M_U_G).max(), 1e-10, "3.1")

### 3.2 The temperature constants (`T_constants`)

T = (γ−1) μ m_p u / k_B. fake_spectra's m_p and k_B (1.67262178×10⁻²⁴ g and 1.38066×10⁻¹⁶ erg/K) differ from CODATA 2018 in the 6th significant digit.

**Its cost.**
- **Per cell:** T is 8.1×10⁻⁶ higher in SkewerJAX. That is negligible for H I. It matters only through steep ion-fraction tables, at the 10⁻⁴ level for Si III near its collisional cut-off (§3.7).
- **Matched mode:** `T_constants="fake_spectra"`.

In [ ]:
rT = np.asarray(P_def["T"]) / np.asarray(S.properties(gp.HI, E_DEF.replace(T_constants="fake_spectra"))["T"])
pred = (phys.mp / phys.kB_erg_per_K) / (1.67262178e-24 / 1.38066e-16)
print(f"T(CODATA) / T(fake_spectra constants) = {np.median(rT):.9f} (predicted {pred:.9f})")
check("T constants: per-cell ratio as predicted", np.abs(rT - pred).max(), 1e-12, "3.2")

### 3.3 The line-width constants (`b_constants`)

The profile's Doppler parameter is b = √(2k_BT/m). SkewerJAX uses CODATA k_B and m = A m_u. fake_spectra's C engine uses its own k = 1.3806504×10⁻¹⁶ and m = A m_p. At equal T,

(b_fs/b_sjx)² = (k_fs/m_p)/(k_B/m_u) = 0.992777 (`gas_physics.FAKE_SPECTRA_B2_RATIO`).

**How it is matched.** The matched mode multiplies T by this ratio inside the profile only (`b_constants="fake_spectra"`), leaving the fields themselves unchanged. It is a validation device: every line is 0.36% narrower than the physical one.

**Its cost.** No field changes, only line shapes; it is measured in the spectrum in §4.

In [ ]:
check("b constants: (b_fs / b_sjx)^2 == FAKE_SPECTRA_B2_RATIO", abs((1.3806504e-16 / 1.67262178e-24) / (phys.kB_erg_per_K / gp.M_U_G) - gp.FAKE_SPECTRA_B2_RATIO), 1e-6, "3.3")
print(f"b_fs / b_sjx = {np.sqrt(gp.FAKE_SPECTRA_B2_RATIO):.6f} for every ion")

### 3.4 The neutral fraction (`x_HI_model`)

**What the snapshot stores.** TNG stores `NeutralHydrogenAbundance` from its own ionisation equilibrium. It includes self-shielding and AGN radiation, and it is consistent with the heating that set T (`3_fields_from_simulations.ipynb`, §3). Both codes keep it for ordinary gas.

**Where they differ: star-forming gas**, whose effective temperature is unphysical. Both recompute x_HI there as the cold, self-shielded phase at 10⁴ K, but with different rules:
- **SkewerJAX** (`"stored_sf"`): cells with SFR > 0, at their own n_H, with the simulation's own UV background rate Γ_HI.
- **fake_spectra** (`sf_neutral=True`, its default; `"fake_sf"`): cells above a density threshold in ρ/m_p, at density ρ/m_p, with a tabulated Γ_HI.

**Check of the emulation.** SkewerJAX's `"fake_sf"` is compared with fake_spectra's own values on 2000 star-forming cells of one snapshot file. The cost is the ratio of the two x_HI on those cells.

In [ ]:
raw_sf = rd.read_rows(0, ["StarFormationRate"])
rows_sf = np.nonzero(raw_sf["StarFormationRate"] > 0)[0][::50][:2000]
Csf = rd.to_gas_cells(rd.read_rows(0, list(sr.GAS_DATASETS), rows=rows_sf), ids=rows_sf)      # file 0: id = row
fv_sf = FS.get("cells_file0_sf_H1_1", lambda: FS.drv.cell_values(FS.spectra_object("single", True), np.array([rd.files[0].name] * rows_sf.size), rows_sf, "H", 1))
x_fakesf = np.asarray(gp.ion_properties(Csf, gp.HI, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf")["ion_fraction"])
x_storedsf = np.asarray(gp.ion_properties(Csf, gp.HI)["ion_fraction"])
x_stored = np.asarray(Csf.fields["x_HI_stored"])
check("x_HI_model='fake_sf' == fake_spectra's sf_neutral=True, 2000 star-forming cells", rel(x_fakesf, fv_sf["ion_frac"]).max(), 5e-5, "3.4",
      detail="(fake_spectra evaluates its formula in float32)")
applied = fv_sf["ion_frac"] != fv_sf["raw_HI"]                             # the cells where fake_spectra applied its rule
if FS.live:                                                                 # the float32 floor: fake_spectra's own formula replayed
    sys.path.insert(0, str(FAKE_SPECTRA_BUILD)); from fake_spectra import gas_properties as _fgp
    _g = object.__new__(_fgp.GasProperties); _g.gamma_UVB, _g.f_bar = FC["gamma_UVB"], 0.17
    _g.gray_opac = float(np.interp(H.redshift, np.arange(9.0), gp._RAHMATI_GRAY))
    x32, x64 = _g._neutral_fraction(fv_sf["rhoH"], 1e4), _g._neutral_fraction(fv_sf["rhoH"].astype(float), 1e4)
    check("fake_spectra's own formula on its float32 inputs reproduces its x_HI", rel(x32[applied], fv_sf["ion_frac"][applied]).max(), 0, "3.4")
    print(f"the same formula in float64: max |rel| {rel(x64[applied], fv_sf['ion_frac'][applied]).max():.2e} "
          f"(SkewerJAX's emulation, float64: {rel(x_fakesf[applied], fv_sf['ion_frac'][applied]).max():.2e}) -> the residual is fake_spectra's float32 arithmetic")
r = x_storedsf / x_fakesf
print(f"star-forming cells: x_HI stored (eff. EOS) median {np.median(x_stored):.3f} | SkewerJAX {np.median(x_storedsf):.3f} | fake_spectra {np.median(x_fakesf):.3f}")
print(f"SkewerJAX / fake_spectra on star-forming cells: median {np.median(r):.4f}, 5-95% {np.percentile(r, 5):.4f}-{np.percentile(r, 95):.4f}; "
      f"cells that fake_spectra leaves at the stored value: {np.mean(np.isclose(x_fakesf, x_stored)):.1%}")
n_sf = {}
for name, S_ in SK.items():
    sf_ = np.nonzero(np.asarray(S_.cells.fields["sfr"]) > 0)[0]
    n_sf[name] = (sf_.size, int(np.isin(sf_, S_.segments("tophat").cell).sum()), int(np.isin(sf_, S_.segments("voronoi").cell).sum()))
print("star-forming cells (selected, touched by a top-hat sphere, crossed as a Voronoi cell):", n_sf)
check("no star-forming cell is touched or crossed by these skewers: the x_HI difference is exactly zero on them",
      float(sum(t[1] + t[2] for t in n_sf.values())), 0, "3.4")
check("fake_spectra sf_neutral=True == False on the 16 skewers (its own runs)",
      np.abs(FS.run("nset_matched", "nset")["tau"] - FS_TAU["nset"]).max(), 0, "3.4")
nH_sf = np.asarray(gp.ion_properties(Csf, gp.HI)["n_H"])
fig, ax = plt.subplots(figsize=(13, 7.5), layout="constrained")
ax.loglog(nH_sf, x_stored, "s", ms=4, color=TOL["grey"], label="stored (effective EOS)")
ax.loglog(nH_sf, x_fakesf, "^", ms=6, mfc="none", color=EC["fake_spectra"], mec="0.35", label=r"fake_spectra: $\rho/m_p$ > threshold, tabulated $\Gamma_{\rm HI}$")
ax.loglog(nH_sf, x_storedsf, "o", ms=4, color=EC["HI"], label=r"SkewerJAX: SFR > 0, at $n_{\rm H}$, the simulation's $\Gamma_{\rm HI}$")
ax.set(xlabel=r"$n_{\rm H}$ [cm$^{-3}$]", ylabel=r"$x_{\rm HI}$", ylim=(1e-3, 2),
       title="Star-forming cells: both rules give a nearly neutral cold phase")
ax.legend(loc="lower right", markerscale=2); plt.show()

*Figure — the neutral fraction of star-forming cells.* The stored values (grey squares) follow the effective equation of state and are not physical. Both rules give a nearly neutral cold phase, but they differ near the threshold of shielding, where fake_spectra's density and rate differ from SkewerJAX's.

**On these sightlines the difference is exactly zero.** Three star-forming cells are *selected* near skewer 8, within the selection radius, but none is touched or crossed by a sightline, and fake_spectra's own runs with and without its rule are identical. The 3.7×10⁻⁵ residual of the emulation is fake_spectra's float32 arithmetic: its own formula, replayed in float32 on its inputs, reproduces its values exactly, and in float64 it differs by the same 3.7×10⁻⁵. It would matter for sightlines through the discs of galaxies.

### 3.5 Selection

Neither code changes the gas: they decide which cells may contribute.
- **fake_spectra** keeps the cells whose top-hat sphere touches the sightline.
- **SkewerJAX** keeps every cell whose generating point is within 2.5 h + Δs/2 of a path point (k-d tree), a superset that also contains every Voronoi cell the path crosses.

Check: SkewerJAX's top-hat cells are exactly the cells that fake_spectra's engine adds to the sightline.

In [ ]:
fs_touch = FS.get("touch_single", lambda: dict(touch=FS.drv.touches(FS.spectra_object("single"), fv, FS.PI, 0)))["touch"].astype(bool)
sjx_touch = np.zeros(len(S.cells), bool); sjx_touch[touch] = True
print(f"single skewer: {len(S.cells):,} cells selected; top hat touches {sjx_touch.sum()} (SkewerJAX) and {fs_touch.sum()} (fake_spectra); "
      f"in only one of the two sets: {np.sum(sjx_touch ^ fs_touch)}")
check("top-hat cells == the cells fake_spectra adds to the sightline", float(np.sum(sjx_touch ^ fs_touch)), 0, "3.5")
VO1 = S.segments("voronoi")
vcells = np.unique(VO1.cell[VO1.cell >= 0])
print(f"Voronoi cells crossed by the skewer: {vcells.size}; of them outside the top-hat set: {np.setdiff1d(vcells, touch).size} "
      f"(cells the path crosses although their sphere V^(1/3) does not reach it)")

**Cost: none.** Both selections are complete for their geometry, so the selection is not a difference between the codes; the geometry is (§3.6).

### 3.6 The deposition geometry (`deposition`): Voronoi cells against top-hat spheres

**What fake_spectra does.** It replaces every cell by a sphere of radius h = V^(1/3) at density (3/4π) n. The sphere has the cell's volume, but it extends further than the cell. Spheres overlap: on average 4.2 cover each point of space. Along the sightline, each sphere's chord gets its column, at its own T and v_pec.

**SkewerJAX's default.** The snapshot's `Coordinates` are the mesh-generating points, so the cells are known exactly: the Voronoi deposition puts the gas where the simulation had it, one cell at a time.

**Matched mode:** `deposition="tophat"`. SkewerJAX computes the exact top-hat chord of every sphere in every segment; fake_spectra's own 7-point quadrature of the chord is part of the integration and is treated in the integration notebook.

**Checks.**
- The top-hat column per position pixel must reproduce fake_spectra's `get_col_density`, and its column-weighted T per pixel fake_spectra's `get_temp`.
- fake_spectra also has an optional Voronoi reconstruction (`kernel="voronoi"`), against which SkewerJAX's Voronoi columns are compared.

In [ ]:
fs1 = FS.run("single_matched", "single", products="tau,colden,temp,velocity"); fs_vor = FS.run("single_matched_voronoi", "single", kernel="voronoi")
nb = v.size
N_th, N_vo = S.column_per_pixel(gp.HI, E_FS, nb)[0], S.column_per_pixel(gp.HI, E_FS.replace(deposition="voronoi"), nb)[0]
T_th, T_vo = S.column_per_pixel(gp.HI, E_FS, nb, weight="T")[0], S.column_per_pixel(gp.HI, E_FS.replace(deposition="voronoi"), nb, weight="T")[0]
has = fs1["colden"][0] > 1e-6 * fs1["colden"][0].max()
check("top hat: total N_HI == fake_spectra", rel(N_th.sum(), fs1["colden"][0].sum()).max(), 1e-6, "3.6")
check("top hat: N_HI per position pixel == fake_spectra", rel(N_th, fs1["colden"][0], has).max(), 1e-4, "3.6", detail=f"(median {np.median(rel(N_th, fs1['colden'][0], has)):.1e})")
check("top hat: column-weighted T per pixel == fake_spectra's get_temp", rel(T_th, fs1["temp"][0], has).max(), 1e-4, "3.6")
check("Voronoi: total N_HI == fake_spectra's Voronoi reconstruction", rel(N_vo.sum(), fs_vor["colden"][0].sum()).max(), 1e-4, "3.6")
hv = fs_vor["colden"][0] > 1e-3 * fs_vor["colden"][0].max()
dN = N_vo / np.maximum(N_th, 1e-300) - 1
print(f"Voronoi vs fake_spectra's Voronoi reconstruction, per pixel: median {np.median(rel(N_vo, fs_vor['colden'][0], hv)):.1e}")
print(f"Voronoi vs top hat, per position pixel: |dN/N| median {np.median(np.abs(dN[has])):.2f}, 90% {np.percentile(np.abs(dN[has]), 90):.2f}; "
      f"column-weighted T: median |dT/T| {np.median(rel(T_vo, T_th, has)):.2f}; total N: {N_vo.sum() / N_th.sum() - 1:+.1e}")
s_pts = S.s[0]; win = (s_pts > 30.0) & (s_pts < 33.0)
nH_cells = np.asarray(P_fs["n_ion"])
segv = dep.segment_fields(VO1, nH_cells, P_fs["T"], np.zeros(VO1.cell.shape))
xpix = (np.arange(nb) + 0.5) * L / nb; wp = (xpix > 30) & (xpix < 33)
pv = dep.tophat_point_values(S.points, S.cells.positions, S.cells.size_cMpc, L, nH_cells, P_fs["T"], lambda c, p: np.zeros(np.shape(c)))
fig, axs = plt.subplots(3, 1, figsize=(16, 15), sharex=True, layout="constrained", gridspec_kw={"height_ratios": [3, 1.3, 2]})
for kk in range(TH1.cell.shape[-1]):
    c = TH1.cell[0, :, kk]; live = (c >= 0) & win
    axs[0].semilogy(s_pts[live], dep.TOPHAT_NORM * nH_cells[c[live]], ".", ms=3, color=EC["tophat"], alpha=0.6)
axs[0].plot([], [], ".", ms=10, color=EC["tophat"], label=r"top hat: each sphere, $(3/4\pi)\,n_c$")
axs[0].semilogy(s_pts[win], pv["n"][0][win], color=EC["tophat"], ls="--", lw=2.5, label="top hat: sum of the overlapping spheres (fake_spectra's geometry)")
axs[0].semilogy(s_pts[win], (segv["column_length"][0] / S.line_elements[0])[win], color=EC["voronoi"], lw=2.5, label="Voronoi: the simulation's cells (SkewerJAX default)")
axs[0].set(ylabel=r"$n_{\rm HI}$ [cm$^{-3}$]", title="The two geometries put the same gas differently along the path"); axs[0].legend(loc="lower right", fontsize=14)
axs[1].plot(s_pts[win], VO1.n_components[0][win], color=EC["voronoi"], lw=2, label="Voronoi pieces per segment")
axs[1].plot(s_pts[win], TH1.n_components[0][win], color=EC["tophat"], ls="--", lw=2, label="top-hat spheres per segment"); axs[1].set(ylabel="components")
axs[1].legend(loc="upper right", ncols=2, fontsize=14)
axs[2].semilogy(xpix[wp], fs1["colden"][0][wp], color=EC["fake_spectra"], **MS["reference"], label="fake_spectra, top hat")
axs[2].semilogy(xpix[wp], N_th[wp], color=EC["tophat"], **MS["matched"], label="SkewerJAX, top hat (matched)")
axs[2].semilogy(xpix[wp], fs_vor["colden"][0][wp], color="0.35", lw=5, alpha=0.35, label="fake_spectra, Voronoi reconstruction")
axs[2].semilogy(xpix[wp], N_vo[wp], color=EC["voronoi"], **MS["default"], label="SkewerJAX, Voronoi (default)")
axs[2].set(xlabel="path length s [cMpc]", ylabel=r"N per 1 km/s pixel [cm$^{-2}$]", title="Each code matches the other in each geometry; the geometries differ by tens of %")
axs[2].legend(loc="lower right", ncols=2, fontsize=14)
add_panel_letters(axs)
plt.show()

*Figure — the two geometries on the same gas.*
- **Top:** the Voronoi field is piecewise constant (blue, solid), one cell at a time. The top hat spreads each cell over a sphere of radius V^(1/3) (yellow dots, one per sphere and segment). Their sum (yellow, dashed) is smooth, with no cell faces.
- **Middle:** 1–2 Voronoi pieces per segment, against up to a dozen overlapping spheres.
- **Bottom:** the column per 1 km/s position pixel. SkewerJAX's top hat lies on fake_spectra's (matched), and its Voronoi columns lie on fake_spectra's own Voronoi reconstruction. The two geometries differ by tens of per cent per pixel.

**Its cost.** The total column is nearly the same: both conserve mass, and the residual comes from the spheres of cells off the sightline. But the column is redistributed along the sightline by tens of per cent per pixel, on the scale of a cell (TNG50-4's cells are 15–30 km/s long), and the column-weighted temperature per pixel changes with it. This is the largest extraction difference on this snapshot (§4).

fake_spectra's Voronoi reconstruction is not used as the reference for the default because it only searches cells whose sphere touches the line, and it samples the line on a fixed grid; its spectra still use the 7-point chord quadrature.

### 3.7 A metal ion: the ion-fraction table (`cloudy_density`, `cloudy_interpolation`)

Both codes read the same Si III fraction table (byte-identical files), but they look it up differently:
- **Density coordinate.** The table's density is the hydrogen density, so SkewerJAX passes n_H. fake_spectra passes ρ/m_p × 0.774132, a fixed hydrogen fraction for the table's metal mixture. The two differ by X_H m_u/(0.774132 × 1.00794 m_p) ≈ 0.98 for X_H = 0.76.
- **Interpolation.** SkewerJAX interpolates log f bilinearly; fake_spectra uses a cubic spline, which overshoots near the "log 0" floor of the table.

Per-cell check on the cells touching skewer 8 (the Si III system), then each difference alone.

In [ ]:
S8 = SK["sk8"]; TH8 = S8.segments("tophat"); t8 = np.unique(TH8.cell[TH8.cell >= 0])
fv8 = FS.cell_values(rd, S8.cells, "sk8", "Si", 3)
f_fs = np.asarray(fv8["ion_frac"])[t8]; nSi_fs = (np.asarray(fv8["elem_den"]) / FC["rscale"])[t8]
live = (f_fs > 1e-8) & (nSi_fs > 0)
Pm8 = S8.properties(SI, E_FS)
check("matched per-cell f_SiIII == fake_spectra (cells touching skewer 8, f > 1e-8)", rel(np.asarray(Pm8["ion_fraction"])[t8], f_fs, live).max(), 5e-5, "3.7",
      detail=f"({live.sum()} cells; fake_spectra's float32 table lookup sets the floor)")
check("matched per-cell n_SiIII == fake_spectra", rel(np.asarray(Pm8["n_ion"])[t8], nSi_fs, live).max(), 5e-5, "3.7")
col_w = (nSi_fs * np.asarray(S8.cells.size_cMpc)[t8])[live]
print(f"{'switched alone, from fake_spectra':34s} {'f: median |rel|':>16s} {'max':>9s} {'n_SiIII, column-weighted':>25s}")
f0, n0 = np.asarray(Pm8["ion_fraction"])[t8][live], np.asarray(Pm8["n_ion"])[t8][live]
for lab, d in (("cloudy_density = hydrogen", dict(cloudy_density="hydrogen")), ("cloudy_interpolation = linear", dict(cloudy_interpolation="linear")),
               ("T_constants = codata", dict(T_constants="codata")), ("mass_unit = amu", dict(mass_unit="amu")), ("all (= default)", {})):
    P = S8.properties(SI, E_FS.replace(**d) if d else E_DEF)
    f_, n_ = np.asarray(P["ion_fraction"])[t8][live], np.asarray(P["n_ion"])[t8][live]
    print(f"{lab:34s} {np.median(np.abs(f_ / f0 - 1)):16.2e} {np.abs(f_ / f0 - 1).max():9.2e} {np.sum(col_w * (n_ / n0 - 1)) / col_w.sum():25.3e}")
CL = S8._cloudy
lg = np.linspace(-7.5, 4.0, 2301)
Pd8 = S8.properties(SI, E_DEF)
fl8 = np.asarray(CL.fraction(SI, np.asarray(Pd8["n_H"])[t8], np.asarray(Pd8["T"])[t8]))
fc8 = np.asarray(CL.fraction(SI, np.asarray(Pd8["n_H"])[t8], np.asarray(Pd8["T"])[t8], interpolation="cubic"))
fig, axs = plt.subplots(1, 2, figsize=(20, 7.5), layout="constrained")
for T_, c in ((10 ** 4.0, TOL["green"]), (10 ** 4.6, TOL["purple"])):
    axs[0].semilogy(lg, np.asarray(CL.fraction(SI, 10 ** lg, np.full_like(lg, T_), interpolation="cubic")), color=c, **MS["matched"], label=f"cubic (fake_spectra), T = {T_:.0e} K")
    axs[0].semilogy(lg, np.asarray(CL.fraction(SI, 10 ** lg, np.full_like(lg, T_))), color=c, **MS["default"], label=f"bilinear (SkewerJAX), T = {T_:.0e} K")
jT = np.argmin(np.abs(CL.log_temp - 4.0)); axs[0].semilogy(CL.log_dens, 10 ** CL.log_fraction_grid(SI)[:, jT], "ko", ms=5, label="table nodes, T = 1e4 K")
axs[0].set(xlabel=r"log$_{10}$ hden [cm$^{-3}$]", ylabel=r"$f_{\rm SiIII}$", ylim=(1e-30, 3), title="The cubic spline rings near the table floor")
axs[0].legend(loc="lower right", fontsize=13)
axs[1].loglog(np.asarray(Pd8["n_H"])[t8], np.abs(fc8 / np.maximum(fl8, 1e-300) - 1), "o", ms=4, color=EC["SiIII"])
axs[1].set(xlabel=r"$n_{\rm H}$ [cm$^{-3}$]", ylabel="|cubic / bilinear − 1|", title="Cells touching skewer 8: a few % where Si III is abundant")
add_panel_letters(axs)
plt.show()

*Figure — left:* the cubic spline (dashed) rings between the nodes where the table drops towards its floor, by orders of magnitude at low density; bilinear interpolation (solid) stays between the nodes. *Right:* on the gas of skewer 8 the two agree to a few per cent where Si III is abundant, and differ far more only where it is negligible.

**Costs.**
- The **density convention** moves f by 10% per cell (median; up to 80%), in either direction depending on the local slope of the table. Weighted by the column it adds +2.1% to n_SiIII.
- The **interpolation** moves f by 7% per cell (median; up to a factor 2), but only −0.3% in the column-weighted sum.
- The mass constant adds its +0.73% and the T constants 10⁻⁵. Together the four make the default's column-weighted n_SiIII +1.9% above fake_spectra's (last row of the table).

**Which is physically correct?** For the density, hden = n_H: that is what the table's density is, while 0.774 is a fixed composition that matches no actual cell. For the interpolation, bilinear in log f: it is monotone between the nodes.

## 4. What each difference costs in the spectrum

The fields above enter τ. To measure each extraction difference in the spectrum, the **integration is held at SkewerJAX's default** (`IntegrationOptions()`: exact frame, every piece's full profile) and the extraction is switched from fake_spectra's to SkewerJAX's:
- one difference at a time, from fake_spectra's options (**isolated**);
- cumulatively (**chain**), which telescopes exactly from [fake_spectra extraction, default integration] to [default extraction, default integration].

Measured on the 16 skewers (Lyα) and on skewer 8 (Si III). EW changes are relative to fake_spectra's own equivalent width. "P1D" is the median |ΔP/P| in two bands of k [s/km].

In [ ]:
EFF = {}
for name, ion, lb in (("nset", gp.HI, LYA), ("sk8", SI, LB_SI)):
    EFF[name] = ss.run_effects(SK[name], ion, lb, v, ss.EXTRACTION_EFFECTS, E_FS, I_DEF)
    ref = FS_TAU["nset"] if name == "nset" else FS_TAU["sk8_SiIII"]
    rows_, tot = ss.effect_rows(EFF[name], dv, ref)
    ss.print_effect_rows(rows_, tot, f"\n{ion.species}, {name}: extraction differences with the default integration")
    EFF[name]["rows"], EFF[name]["tot"] = rows_, tot
    x_end, i_end = EFF[name]["options"][list(EFF[name]["links"])[-1]]
    check(f"{ion.species}: chain end == SkewerJAX default (extraction and integration)", float((ss.relevant_options(x_end, ion), i_end) != (ss.relevant_options(E_DEF, ion), I_DEF)), 0, "4")
    check(f"{ion.species}: extraction total = sum of the isolated differences, d<F>", abs(tot["dF"] - tot["dF_sum"]), 0.1 * tot["dF_abs"] + 1e-6, "4", kind="explained")
    check(f"{ion.species}: extraction total = sum of the isolated differences, dEW/EW", abs(tot["dEW"] - tot["dEW_sum"]), 0.1 * tot["dEW_abs"] + 1e-6, "4", kind="explained")

In [ ]:
R = EFF["nset"]; labs = [r_["effect"] for r_ in R["rows"]]
S1 = SK["single"]
R1 = ss.run_effects(S1, gp.HI, LYA, v, ss.EXTRACTION_EFFECTS, E_FS, I_DEF, verbose=False)
L1 = list(R1["links"])
win = (v > 1200) & (v < 1700)
dmat = np.array([np.exp(-R1["links"][L1[i]][0]) - np.exp(-R1["links"][L1[i - 1]][0]) for i in range(1, len(L1))])
y = np.arange(len(labs))
fig, axs = plt.subplots(1, 2, figsize=(20, 7.5), layout="constrained")
for ax_, key, xl in ((axs[0], "dF", r"$\Delta\langle F\rangle$"), (axs[1], "dEW", r"$\Delta EW/EW$")):
    ax_.barh(y + 0.2, [r_[f"{key}_chain"] for r_ in R["rows"]], 0.4, color=TOL["blue"], label="along the chain")
    ax_.barh(y - 0.2, [r_[f"{key}_iso"] for r_ in R["rows"]], 0.4, color=TOL["red"], hatch="//", edgecolor="k", label="isolated")
    ax_.set_yticks(y, labs if ax_ is axs[0] else [""] * len(labs)); ax_.invert_yaxis(); ax_.axvline(0, color="k", lw=1.2)
    ax_.set_xscale("symlog", linthresh=1e-6); ax_.set(xlabel=xl)
fig.suptitle(r"Ly$\alpha$, 16 skewers: the Voronoi geometry and the mass constant dominate the extraction difference", fontsize=22)
fig.legend(*axs[0].get_legend_handles_labels(), loc="outside lower center", ncols=2)
add_panel_letters(axs, x=0.0, y=1.07)
plt.show()
lim = np.abs(dmat[:, win]).max()
fig, axs = plt.subplots(2, 1, figsize=(16, 11), sharex=True, layout="constrained", gridspec_kw={"height_ratios": [1.2, 2]})
axs[0].plot(v[win], np.exp(-R1["links"]["start"][0])[win], color=EC["HI"], **MS["matched"], label="fake_spectra's extraction")
axs[0].plot(v[win], np.exp(-R1["links"][L1[-1]][0])[win], color=EC["HI"], **MS["default"], label="SkewerJAX's extraction")
axs[0].set(ylabel="F", ylim=(-0.05, 1.05), title="Single skewer, default integration: the two extractions"); axs[0].legend(loc="upper center", ncols=2)
im = axs[1].imshow(dmat[:, win], aspect="auto", cmap=tol_cmap("BuRd"), interpolation="nearest", extent=(v[win][0], v[win][-1], len(L1) - 1.5, -0.5),
                   norm=mpl.colors.SymLogNorm(1e-5, vmin=-lim, vmax=lim))
axs[1].set_yticks(range(len(L1) - 1), L1[1:]); axs[1].set(xlabel="v [km/s]", title="ΔF of each step of the chain (red: more transmission)")
fig.colorbar(im, ax=axs[1], label="F(step) − F(previous)")
add_panel_letters(axs)
plt.show()

*Figures — the extraction differences in the spectrum.* Bars: the change in ⟨F⟩ and in the equivalent width, along the chain and isolated (symmetric-log axis). Map: where along the single skewer each step changes F (red: more transmission).

## 5. Summary: the extraction differences, ranked

Ranked by the rms change in F of the isolated switch on the 16 skewers (Lyα; for Si III, skewer 8). "Per cell" and "along the skewer" summarise §3.

In [ ]:
PER_CELL = {"T constants": "T +8.1e-6", "b constants": "b +0.36% (all lines)", "mass constant": "n +0.728% (all elements)",
            "neutral fraction": "x_HI of SF cells only (none here)", "CLOUDY density": "f_SiIII: median 10%", "CLOUDY interpolation": "f_SiIII: median 7%, up to x2",
            "deposition": "none (same cells)"}
ALONG = {"T constants": "T per pixel +8.1e-6", "b constants": "line widths", "mass constant": "N per pixel +0.728%", "neutral fraction": "0 on these skewers",
         "CLOUDY density": "N_SiIII +2.1% (col.-weighted)", "CLOUDY interpolation": "N_SiIII -0.3% (col.-weighted)", "deposition": "N per pixel: tens of % (cell scale)"}
print(f"{'rank':4s} {'difference':22s} {'ion':6s} {'rms dF':>9s} {'max |dF|':>9s} {'d<F>':>10s} {'dEW/EW':>10s}  {'per cell':30s} {'along the skewer':34s} matched mode")
allrows = [(r_, "HI") for r_ in EFF["nset"]["rows"]] + [(r_, "SiIII") for r_ in EFF["sk8"]["rows"] if r_["effect"].startswith("CLOUDY")]
for i, (r_, ion) in enumerate(sorted(allrows, key=lambda x: -x[0]["rms_iso"]), 1):
    e = next(x for x in ss.EXTRACTION_EFFECTS if x.label == r_["effect"])
    mm = ", ".join(f"{k}={getattr(E_FS, k)!r}" for k in e.change)
    print(f"{i:4d} {r_['effect']:22s} {ion:6s} {r_['rms_iso']:9.2e} {r_['max_iso']:9.2e} {r_['dF_iso']:10.2e} {r_['dEW_iso']:10.2e}  {PER_CELL[r_['effect']]:30s} {ALONG[r_['effect']]:34s} {mm}")
t_ = EFF["nset"]["tot"]
print(f"\nLyα, 16 skewers: extraction total (default - fake_spectra extraction, default integration): d<F> = {t_['dF']:+.4e}, dEW/EW = {t_['dEW']:+.4e}")
if EXPORT_REFERENCE and FS.live:
    print(f"reference file {FS_REFERENCE.name}: {FS.export() / 1e6:.1f} MB")

## 6. Summary of the checks

In [ ]:
import collections, resource
print(f"{'tag':12s} {'section':7s} name")
for r in RESULTS:
    print(f"{r['tag']:12s} {r['section']:7s} {r['name']}  ({r['value']:.2e} {'>' if r['kind'] == 'expect' else '<='} {r['tol']:g}) {r['detail']}")
print("\nSUMMARY:", dict(collections.Counter(r["tag"] for r in RESULTS)), f"of {len(RESULTS)} checks")
print(f"wall time {time.perf_counter() - T_START:.0f} s | peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")